# Семинар 1. Линейная регрессия

#### Шаг 1. Загрузка датасета и вывод на экран

In [ ]:
import pandas as pd

# Загружаем CSV-файл с данными
df = pd.read_csv('data.csv')

# Выводим первые 5 строк на экран
print("Шаг 1: Первые 5 строк датасета:")
print(df.head())

#### Шаг2. Разделение выборки на тренировочную, валидационную и тестовую с выбором нужных столбцов:
* в тренировочной: 300 первых заказов;
* в валидационной: 100 следующих; 
* в тестовой: 100 последних.

In [ ]:
# 1. Выбираем целевой столбец (то, что предсказываем)
TARGET_COLUMN = 'Время доставки (в минутах)'

# 2. Удаляем ненужные столбцы (текстовые и идентификаторы)
df_clean = df.drop(columns=['Дата заказа (ГГГГ-ММ-ДД)', 'Шифр заказа (ID)'])

# 3. Разделяем выборки по строкам (по условию задания)
train_df = df_clean.iloc[:300]       # 300 первых заказов для обучения
val_df   = df_clean.iloc[300:400]    # 100 следующих для валидации
test_df  = df_clean.iloc[400:500]    # 100 последних для итогового теста

# 4. Отделяем факторы (X) от ответов (y)
X_train = train_df.drop(columns=[TARGET_COLUMN])
y_train = train_df[TARGET_COLUMN]

X_val = val_df.drop(columns=[TARGET_COLUMN])
y_val = val_df[TARGET_COLUMN]

X_test = test_df.drop(columns=[TARGET_COLUMN])
y_test = test_df[TARGET_COLUMN]

print("Шаг 2 выполнен. Использованы столбцы признаков:", list(X_train.columns))

#### Вопрос 1. 

Для чего мы разбиваем данные на выборки? Для чего нам нужна тренировачная, для чего валидационная, для чего тестовая?

На тренировачной она учится, на валидацонной закрепляет материал, а на тестовой уже проверяет свои навыки. Это делается для того, чтобы правильно модель обучить и порседством проверк убедиться, что всё нормально

#### Шаг 3. Установка sklearn чтобы обучать модель линейной регрессии через Ridge

In [ ]:
pip install scikit-learn numpy matplotlib pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### Шаг 4. Обучение модели линейной регрессии на тренировочной выборке с помощью Ridge и sparse_cg

In [ ]:
# Списки для сохранения ошибок
train_sse, train_mse, train_rmse = [], [], []
val_sse, val_mse, val_rmse = [], [], []

max_iterations = 20  # Число итераций (шагов)

# Цикл обучения от 1 до 20 итераций
for i in range(1, max_iterations + 1):
    # Обучаем модель с параметром solver='sparse_cg' и max_iter=i
    model = Ridge(solver='sparse_cg', max_iter=i, random_state=42)
    model.fit(X_train, y_train)
    
    # Делаем предсказания
    pred_train = model.predict(X_train)
    pred_val   = model.predict(X_val)
    
    # Считаем MSE
    mse_tr = mean_squared_error(y_train, pred_train)
    mse_v  = mean_squared_error(y_val, pred_val)
    
    # Сохраняем MSE, RMSE и SSE
    train_mse.append(mse_tr)
    val_mse.append(mse_v)
    
    train_rmse.append(np.sqrt(mse_tr))
    val_rmse.append(np.sqrt(mse_v))
    
    train_sse.append(mse_tr * len(y_train))
    val_sse.append(mse_v * len(y_val))

#### Шаг 5. Вывод на экран графиков ошибок (SSE, MSE, RMSE) в зависимости от итераций для train и val выборок.

In [ ]:
# Строим графики ошибок
plt.figure(figsize=(16, 5))

# График 1: SSE
plt.subplot(1, 3, 1)
plt.plot(range(1, max_iterations + 1), train_sse, label='Train SSE', color='blue')
plt.plot(range(1, max_iterations + 1), val_sse, label='Val SSE', color='orange')
plt.title('SSE (Сумма квадратов ошибок)')
plt.xlabel('Итерации')
plt.ylabel('Ошибка')
plt.legend()
plt.grid(True)

# График 2: MSE
plt.subplot(1, 3, 2)
plt.plot(range(1, max_iterations + 1), train_mse, label='Train MSE', color='blue')
plt.plot(range(1, max_iterations + 1), val_mse, label='Val MSE', color='orange')
plt.title('MSE (Среднеквадратичная ошибка)')
plt.xlabel('Итерации')
plt.ylabel('Ошибка')
plt.legend()
plt.grid(True)

# График 3: RMSE
plt.subplot(1, 3, 3)
plt.plot(range(1, max_iterations + 1), train_rmse, label='Train RMSE', color='blue')
plt.plot(range(1, max_iterations + 1), val_rmse, label='Val RMSE', color='orange')
plt.title('RMSE (Корень из MSE)')
plt.xlabel('Итерации')
plt.ylabel('Ошибка в минутах')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

#### Вопрос 2.

Какую информацию про нашу модель несут эти графики?

Эти графики показывают, что модель обучилась правильно, так как ошибка падает до определённого значения и дальше идёт по горизонту

#### Вопрос 3. 

Какая траектория линий train и val ошибок будет на графиках недообученной модели и почему?

Очень высокая ошибка, но график всё ещё продолжает идти вниз. Такое происходит, потому что модели дали слишком мало информации и она просто не смогла установить связь между данными.

Какая траектория линий train и val ошибок будет на графиках обученной модели и почему?

График стремится вниз, а затем, достигнув определённой точки, начинает идти ровно по горизонту, такое происходит и на тренировачной, и на валидоционной сборке, так что линии идут параллельно. Такое просиходит из-за того, что разница в ошибке достигает какого-то определённого малого значений, уменьшать, которое нет смысла. На разных алгоритмах они идёт чуть по-разному, где-то тренировачная сборка идёт по графику выше, где-то ниже.

Какая траектория линий train и val ошибок будет на графиках переобученной модели и почему?

На тренировачной сборке ошибка идёт к нулю, а на валидационной она также сначала идёт вниз, а потом идёт вверх, создавая разрыв между линиями. Такое просиходит из-за того, что модель заучила данные из тренировачной сборки и слишком близко к сердцу их приняла, и начали их идеализировать. Из-за чего модель просто перестаёт норамльно работать с новыми данными

#### Шаг 6. Применение модели на тестовой выборке

In [ ]:
# Создаем и обучаем финальную модель на 1000 итераций для полной точности
final_model = Ridge(solver='sparse_cg', max_iter=1000, random_state=42)
final_model.fit(X_train, y_train)

# Применяем модель к тестовым 100 заказам (которые она никогда не видела)
y_test_pred = final_model.predict(X_test)

print("Шаг 6: Предсказания для тестовой выборки успешно сгенерированы.")

#### Шаг 7. RMSE на тестовой выборке

In [ ]:
# Вычисляем финальную ошибку RMSE
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)

print(f"Шаг 7: Финальное значение RMSE на тестовой выборке: {test_rmse:.4f} минут")

#### Вопрос 4.

Что получившийся RMSE может сказать о нашей модели?

Он говорит, что модель правильно обучиась и она имеет высокую точность, в среднем 4,5 минуты